# NPDES Data Cleaning: ATTAINS Assessment-Unit Summaries (Iowa)

Cleans the national NPDES → ATTAINS crosswalk down to Iowa: one tidy row per
(`npdes_id`, `assessment_unit_id`), recording the condition and designated-use
support of the assessment unit (water body) the facility is associated with.

**Input:**  `data/tabular/01_raw/npdes/NPDES_ATTAINS_AU_SUMMARIES.csv` (national)
**Output:** `data/tabular/02_clean/npdes/npdes-attains-clean.csv`

| raw column                 | meaning                                          |
|----------------------------|--------------------------------------------------|
| `NPDES_ID`                 | NPDES permit number (join key)                   |
| `REGISTRY_ID`              | FRS registry id                                  |
| `REPORTINGCYCLE`           | assessment reporting cycle (year)                |
| `ASSESSMENTUNITIDENTIFIER` | ATTAINS assessment-unit id                       |
| `ASSESSMENTUNITNAME`       | water-body name                                  |
| `WATER_CONDITION`          | overall condition (Good / Impaired / Unknown …)  |
| `*_USE`                    | support of each designated use                   |
| `POT_IMP_PARAMETERS`       | potential impairment parameters (pipe-delimited) |
| `CAUSE_GROUPS_IMPAIRED`    | impairment cause groups (pipe-delimited)         |

**Cleaning steps** — filter to Iowa, snake_case the columns, type the reporting
cycle, drop columns that carry no Iowa information, normalize the pipe-delimited
lists, then de-duplicate on (`npdes_id`, `assessment_unit_id`).

In [1]:
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "npdes"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "npdes"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Project scope: Iowa only (STATE == "IA"), filtered after the load.
STATE = "IA"
print("Repo root:", REPO_ROOT)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/npdes


## Step 1 — Load (national) and filter to Iowa

In [2]:
df = pd.read_csv(RAW_DIR / "NPDES_ATTAINS_AU_SUMMARIES.csv", dtype="string")
n_national = len(df)
print(f"Loaded {n_national:,} national rows")
print("Columns:", list(df.columns))

df = df[df["STATE"] == STATE].copy()
print(f"\nFiltered to {STATE}: {len(df):,} rows, {df['NPDES_ID'].nunique()} permits")
print("\nNulls per column (Iowa):")
print(df.isna().sum().to_string())

Loaded 820,292 national rows
Columns: ['REGISTRY_ID', 'ECHO_DFR_URL', 'NPDES_ID', 'REPORTINGCYCLE', 'STATE', 'ASSESSMENTUNITIDENTIFIER', 'AU_URL', 'ASSESSMENTUNITNAME', 'WATER_CONDITION', 'POT_IMP_PARAMETERS', 'E90_POT_IMP_PARAMETERS', 'DRINKINGWATER_USE', 'ECOLOGICAL_USE', 'FISHCONSUMPTION_USE', 'RECREATION_USE', 'OTHER_USE', 'CAUSE_GROUPS_IMPAIRED']

Filtered to IA: 1,102 rows, 1018 permits

Nulls per column (Iowa):
REGISTRY_ID                    0
ECHO_DFR_URL                   0
NPDES_ID                       0
REPORTINGCYCLE                 0
STATE                          0
ASSESSMENTUNITIDENTIFIER       0
AU_URL                         0
ASSESSMENTUNITNAME             0
WATER_CONDITION                0
POT_IMP_PARAMETERS           947
E90_POT_IMP_PARAMETERS      1074
DRINKINGWATER_USE            987
ECOLOGICAL_USE                 0
FISHCONSUMPTION_USE         1102
RECREATION_USE                 0
OTHER_USE                   1050
CAUSE_GROUPS_IMPAIRED        529


## Step 2 — Drop uninformative columns

`STATE` is now constant, and within Iowa some use columns are entirely empty
(no assessments recorded), so they carry no signal. Drop any column that is 100%
null in the Iowa subset, plus the redundant `STATE`.

In [3]:
all_null = [c for c in df.columns if df[c].isna().all()]
print("All-null columns dropped:", all_null)
df = df.drop(columns=["STATE", *all_null])

All-null columns dropped: ['FISHCONSUMPTION_USE']


## Step 3 — Rename to snake_case and type the reporting cycle

In [4]:
RENAME = {
    "NPDES_ID": "npdes_id", "REGISTRY_ID": "registry_id",
    "ECHO_DFR_URL": "echo_dfr_url", "REPORTINGCYCLE": "reporting_cycle",
    "ASSESSMENTUNITIDENTIFIER": "assessment_unit_id", "AU_URL": "au_url",
    "ASSESSMENTUNITNAME": "assessment_unit_name", "WATER_CONDITION": "water_condition",
    "POT_IMP_PARAMETERS": "potential_impairment_parameters",
    "E90_POT_IMP_PARAMETERS": "e90_potential_impairment_parameters",
    "DRINKINGWATER_USE": "drinkingwater_use", "ECOLOGICAL_USE": "ecological_use",
    "FISHCONSUMPTION_USE": "fishconsumption_use", "RECREATION_USE": "recreation_use",
    "OTHER_USE": "other_use", "CAUSE_GROUPS_IMPAIRED": "cause_groups_impaired",
}
df = df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns})
df["reporting_cycle"] = pd.to_numeric(df["reporting_cycle"], errors="coerce").astype("Int64")
print("Reporting cycles:", sorted(df["reporting_cycle"].dropna().unique()))
print("Columns:", list(df.columns))

Reporting cycles: [np.int64(2024)]
Columns: ['registry_id', 'echo_dfr_url', 'npdes_id', 'reporting_cycle', 'assessment_unit_id', 'au_url', 'assessment_unit_name', 'water_condition', 'potential_impairment_parameters', 'e90_potential_impairment_parameters', 'drinkingwater_use', 'ecological_use', 'recreation_use', 'other_use', 'cause_groups_impaired']


## Step 4 — Normalize pipe-delimited lists and use statuses

The parameter / cause-group columns are ` | `-delimited lists; collapse stray
whitespace to a canonical ` | ` separator. Confirm the designated-use columns
only take the expected ATTAINS support vocabulary.

In [5]:
LIST_COLS = [c for c in ["potential_impairment_parameters",
                         "e90_potential_impairment_parameters",
                         "cause_groups_impaired"] if c in df.columns]
for c in LIST_COLS:
    df[c] = (
        df[c].str.split("|").apply(
            lambda parts: " | ".join(p.strip() for p in parts if p.strip())
            if isinstance(parts, list) else parts
        ).replace("", pd.NA)
    )

USE_VOCAB = {"Fully Supporting", "Not Supporting", "Insufficient Information"}
use_cols = [c for c in df.columns if c.endswith("_use")]
for c in use_cols:
    df[c] = df[c].str.strip()
    bad = set(df[c].dropna().unique()) - USE_VOCAB
    assert not bad, f"{c}: unexpected use status {bad}"
print("Use columns:", use_cols)
print(df["water_condition"].value_counts(dropna=False).to_string())

Use columns: ['drinkingwater_use', 'ecological_use', 'recreation_use', 'other_use']
water_condition
Impaired - 303(d) Listed                            397
Unknown                                             370
Good                                                144
Impaired - With Restoration Plan                    105
Impaired - 303(d) Listed - With Restoration Plan     64
Unknown - With Restoration Plan                      12
Impaired                                              7
Good - With Restoration Plan                          3


## Step 5 — De-duplicate, sort, and write

(`npdes_id`, `assessment_unit_id`) identifies a facility/water-body pairing
within the single reporting cycle. Drop exact duplicates, confirm the key, sort,
and write.

In [6]:
df = (
    df.drop_duplicates()
      .sort_values(["npdes_id", "assessment_unit_id"])
      .reset_index(drop=True)
)
dupes = df.duplicated(["npdes_id", "assessment_unit_id"]).sum()
assert dupes == 0, f"{dupes} duplicate (npdes_id, assessment_unit_id) rows"

out_path = CLEAN_DIR / "npdes-attains-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols (from {n_national:,} national) to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 1,102 rows × 15 cols (from 820,292 national) to:
  data/tabular/02_clean/npdes/npdes-attains-clean.csv


,registry_id,echo_dfr_url,npdes_id,reporting_cycle,assessment_unit_id,au_url,assessment_unit_name,water_condition,potential_impairment_parameters,e90_potential_impairment_parameters,drinkingwater_use,ecological_use,recreation_use,other_use,cause_groups_impaired
0,110070244328,https://echo.epa.gov/detailed-facility-report?...,COPIU0087,2024,IA 06-WEM-1708,https://mywaterway.epa.gov/waterbody-report/21...,Missouri River,Impaired - 303(d) Listed,NaN,NaN,<NA>,Not Supporting,Not Supporting,<NA>,HABITAT ALTERATIONS | HYDROLOGIC ALTERATION | ...
1,110036385104,https://echo.epa.gov/detailed-facility-report?...,IA0000035,2024,IA 04-LDM-1010,https://mywaterway.epa.gov/waterbody-report/21...,Des Moines River,Impaired - 303(d) Listed,NaN,NaN,<NA>,Not Supporting,Not Supporting,<NA>,CAUSE UNKNOWN - FISH KILLS | PATHOGENS
2,110045323020,https://echo.epa.gov/detailed-facility-report?...,IA0000051,2024,IA 01-NEM-75,https://mywaterway.epa.gov/waterbody-report/21...,Mississippi River,Good,NaN,NaN,<NA>,Fully Supporting,Fully Supporting,<NA>,NaN
3,110045323020,https://echo.epa.gov/detailed-facility-report?...,IA0000051,2024,IA 01-TRK-131,https://mywaterway.epa.gov/waterbody-report/21...,Little Maquoketa River,Impaired - 303(d) Listed,NaN,NaN,<NA>,Not Supporting,Insufficient Information,<NA>,CAUSE UNKNOWN - IMPAIRED BIOTA
4,110000413954,https://echo.epa.gov/detailed-facility-report?...,IA0000060,2024,IA 02-CED-462,https://mywaterway.epa.gov/waterbody-report/21...,Cedar River,Impaired - 303(d) Listed,NaN,NaN,<NA>,Insufficient Information,Not Supporting,<NA>,PATHOGENS
